In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD01"
archivos = ['agrosavia.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
suelo = pd.read_csv(RAW / "agrosavia.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /home/tuptc/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /home/tuptc/big-data-postgraduate
Salidas: /home/tuptc/big-data-postgraduate/kit/salidas/soluciones/PD01


In [2]:
# 1. Inspección Inicial

# Guardar y mostrar 
filas_iniciales = len(suelo) # N° Filas
columnas_iniciales = len(suelo.columns) # N° Columnas

print(f"Filas totales: {filas_iniciales}")
print(f"Columnas totales: {columnas_iniciales}\n")

# tipos de datos 
print("--- Tipos de columnas ---")
print(suelo.dtypes)

#Mostrar los primeros 5 registros
display(suelo.head(5))

Filas totales: 92738
Columnas totales: 32

--- Tipos de columnas ---
Secuencial                            object
Fecha de Análisis                     object
Departamento                          object
Municipio                             object
Cultivo                               object
Estado                                object
Tiempo de establecimiento             object
Topografia                            object
Drenaje                               object
Riego                                 object
Fertilizantes aplicados               object
pH agua:suelo                         object
Materia organica                      object
Fósforo Bray II                       object
Azufre Fosfato monocalcico            object
Acidez Intercambiable                 object
Aluminio intercambiable               object
Calcio intercambiable                 object
Magnesio intercambiable               object
Potasio intercambiable                object
Sodio intercambiable           

,Secuencial,Fecha de Análisis,Departamento,Municipio,Cultivo,Estado,Tiempo de establecimiento,Topografia,Drenaje,Riego,...,Conductividad electrica,Hierro disponible olsen,Cobre disponible,Manganeso disponible Olsen,Zinc disponible Olsen,Boro disponible,Hierro disponible doble acido,Cobre disponible doble acido,Manganeso disponible doble acido,Zinc disponible doble acido
0,1,7/01/2014,NARIÑO,SAN ANDRÉS DE TUMACO,No Indica,No indica,No indica,No indica,No indica,No indica,...,0.133,66.39,1.700,1.700,1.5,0.187,ND,ND,ND,ND
1,2,21/08/2014,HUILA,SANTA MARÍA,Granadilla,Establecido,De 1 a 5 años,Pendiente,Buen drenaje,No Tiene,...,2.749,250.9,3.7,29.4,28.70,2.084,ND,ND,ND,ND
2,3,22/08/2014,ANTIOQUIA,LIBORINA,Café,Por establecer,No indica,Pendiente,Mal drenaje,No Tiene,...,0.328,390,11.10,8.299,5,0.085,ND,ND,ND,ND
3,4,22/08/2014,ANTIOQUIA,LIBORINA,Maracuyá,Por establecer,No indica,Ondulado,Mal drenaje,No Tiene,...,0.171,200,4.2,4.699,1,0.229,ND,ND,ND,ND
4,5,22/08/2014,ANTIOQUIA,LIBORINA,Café,Por establecer,No indica,Ondulado,Mal drenaje,No Tiene,...,0.323,117.9,3.8,4.600,0.700,0.139,ND,ND,ND,ND


In [3]:
# 2. Diagnostico de 5 campos
# Campos para el diagnostico
campos = ["Secuencial","Fecha de Análisis","Departamento","Municipio","pH agua:suelo",]

registros = []

# Por cada columna en "Campos"
for col in campos:
    s = suelo[col]
    # Numero de nulos
    nulos = s.isna().sum()
    
    # Numero de cadenas cacías o con solo espacios
    vacios = s.str.strip().eq("").sum()
    
    # Cantidad de 'ND'´s (No Determinado / No Disponible)
    nd_count = s.str.strip().str.upper().eq("ND").sum()
    
    registros.append({"campo": col,"nulos": nulos,"vacios": vacios,"No_Disponibles": nd_count,})

# DataFrame "perfil"
perfil = pd.DataFrame(registros)
display(perfil)

,campo,nulos,vacios,No_Disponibles
0,Secuencial,0,0,0
1,Fecha de Análisis,0,0,0
2,Departamento,0,0,0
3,Municipio,0,2,0
4,pH agua:suelo,0,0,0


In [4]:
# 3. Conversión de pH y Fecha

# Para el ph se utiliza to_numeric y estandarizando el uso de "." para separar decimales
ph_limpio = suelo["pH agua:suelo"].str.strip().str.replace(",", ".", regex=False)
suelo["ph"] = pd.to_numeric(ph_limpio, errors="coerce")

# Conversión de Fecha usando to_datetime en formato (%d/%m/%Y)
suelo["fecha_dt"] = pd.to_datetime(suelo["Fecha de Análisis"].str.strip(),format="%d/%m/%Y",errors="coerce",)

# fallos de conversión en ph (textos NO vacíos que terminaron en NaN)
es_vacio_ph = suelo["pH agua:suelo"].str.strip().eq("")
fallos_ph = suelo["ph"].isna() & (~es_vacio_ph)
# fallos de conversión en fecha
es_vacio_fecha = suelo["Fecha de Análisis"].str.strip().eq("")
fallos_fecha = suelo["fecha_dt"].isna() & (~es_vacio_fecha)

# Print de resultados
print(f"Total fallos de conversión en pH (excluyendo vacíos): {fallos_ph.sum()}")
print("Algunos valores que no se convirtieron en pH:")
display(suelo.loc[fallos_ph, ["pH agua:suelo"]].head(5))

print(f"\nFallos de conversión en Fecha (sin contar vacíos): {fallos_fecha.sum()}")
print("Algunos valores no convertibles en Fecha:")
display(suelo.loc[fallos_fecha, ["Fecha de Análisis"]].head(5))

Total fallos de conversión en pH (excluyendo vacíos): 11
Algunos valores que no se convirtieron en pH:


,pH agua:suelo
7282,4..01
46528,>10.00
46530,>10.00
52903,<3.80
56330,<3.80



Fallos de conversión en Fecha (sin contar vacíos): 0
Algunos valores no convertibles en Fecha:


,Fecha de Análisis


In [5]:
# 4. Registros para revisión

# Creación de máscaras para cada motivo de revisión
m_ph_nulo = suelo["ph"].isna() #null?
m_ph_fuera_de_rango = ~suelo["ph"].between(0, 14) & suelo["ph"].notna() #En rango?
m_fecha_nula = suelo["fecha_dt"].isna() #Fecha no interpretada?

# Adición de la columna booleana en_revision (True cuando cumple una o más condiciones descritas en las mascaras)
suelo["en_revision"] = m_ph_nulo | m_ph_fuera_de_rango | m_fecha_nula

# Conteos por motivo
conteo_ph_nulo = m_ph_nulo.sum()
conteo_ph_fuera_de_rango = m_ph_fuera_de_rango.sum()
conteo_fecha_nula = m_fecha_nula.sum()

# Conteo de filas marcadas (en_revision == True) y no marcadas
total_revision = suelo["en_revision"].sum()
total_normales = (~suelo["en_revision"]).sum()

# Display de resultados
print("--- CONTEO DE MOTIVOS DE REVISIÓN ---")
print(f"pH ausente/nulo:                  {conteo_ph_nulo:,}")
print(f"pH fuera del intervalo [0, 14]:  {conteo_ph_fuera_de_rango:,}")
print(f"Fecha no interpretable:           {conteo_fecha_nula:,}")
print(f"-------------------------------------")
print(f"Total filas marcadas en revisión: {total_revision:,}")
print(f"Total filas normales (sin marca): {total_normales:,}")
print(f"Total general (suma):            {total_revision + total_normales:,}")

# Comprobación con assert (conservación del total)
assert (total_revision + total_normales) == len(suelo), "¡Ojo: la suma de marcadas y no marcadas no coincide con el total!"
print("\n ¡Comprobado: Se conservaron todas las filas originales!")

--- CONTEO DE MOTIVOS DE REVISIÓN ---
pH ausente/nulo:                  11
pH fuera del intervalo [0, 14]:  0
Fecha no interpretable:           0
-------------------------------------
Total filas marcadas en revisión: 11
Total filas normales (sin marca): 92,727
Total general (suma):            92,738

 ¡Comprobado: Se conservaron todas las filas originales!


In [6]:
#5 Evidencias y conclusiones
# Definición de la ruta de salida según las instrucciones (carpeta OUT)
archivo_salida = OUT / "controles_agrosavia.csv"

# Construcción de la tabla de controles pedida
controles = pd.DataFrame({
    "control": ["total de entrada","pH sin número","pH fuera del intervalo","fecha no interpretable","filas en revisión"],
    "cantidad": [len(suelo),conteo_ph_nulo,conteo_ph_fuera_de_rango,conteo_fecha_nula,total_revision]})

# Display
print("--- TABLA DE CONTROLES ---")
display(controles)

# Exportación a la carpeta OUT sin índice
controles.to_csv(archivo_salida,index=False,encoding="utf-8-sig")

print(f"\n¡Tabla de controles exportada a: {archivo_salida}!")

--- TABLA DE CONTROLES ---


,control,cantidad
0,total de entrada,92738
1,pH sin número,11
2,pH fuera del intervalo,0
3,fecha no interpretable,0
4,filas en revisión,11



¡Tabla de controles exportada a: /home/tuptc/big-data-postgraduate/kit/salidas/soluciones/PD01/controles_agrosavia.csv!


*Hallazgos y conclusiones:*
- Sobre los datos de pH: En la resolución del taller se encontró consistencia en los datos de pH, donde de las 92.738 observaciones analizadas en el dataset de suelos, únicamente 11 registros presentaron un valor nulo o ausente en el parámetro de pH (esto es menos del 0.02% del total), además de no tener valores fuera del rango válido ([0-14]). Esto muestra que la gran mayoría de mediciones en el archivo original aptas para análisis agronómicos.
- Integridad del registro temporal y conservación de registros: El 100% de los registros de fecha fueron correctamente parseados e interpretados como marcas de tiempo válidas (fecha_dt), sin generación de nulos por errores de formato. Además, la técnica de control utilizada permitió categorizar anomalías mediante la bandera "en_revision" conservando la muestra original sin una pérdida de datos. 